# Imports

In [ ]:
from cytools import Polytope

In [ ]:
import sys; sys.path.append('..')
from src import cydata, Zp, lattice, validation

In [ ]:
import sys; sys.path.append('../../cornell-dev')
from projects.kklt.kklt_lib import kklt_conifolds

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Load manwe

Load from the pickle file associated to the dSv1 paper

In [ ]:
import gzip, pickle
with gzip.open('dSv1.p', 'rb') as f:
    dSv1_df = pickle.load(f)

Construct Manwe

In [ ]:
# Manwe
# -----
# get the CY
manwe_df = dSv1_df[dSv1_df['name']=='manwe'].iloc[0]
p  = Polytope(manwe_df['dual points'])
t  = p.triangulate(heights=manwe_df['mirror heights'])
cy = t.cy()

# get the conifold charge
# -----------------------
conis = list(kklt_conifolds.kklt_conifolds(p.dual(), as_class=True))
assert len(conis) == 1
q = conis[0].conifold_charge()

# set the COB (same one used in paper)
cob = np.array([[0,0,0,0,0,-1,0,0],
[-1,0,0,0,0,0,0,0],
[0,-1,0,0,0,0,0,0],
[0,0,-1,0,0,0,0,0],
[0,0,0,-1,0,0,0,0],
[0,0,0,0,-1,0,0,0],
[0,0,0,0,0,0,-1,0],
[0,0,0,0,0,0,0,-1]])

# make the data object
# --------------------
data = cydata.CYData.from_cy(cy, coni_curve=q, coni_cob=cob)

Test the PFV from the paper

In [ ]:
K = np.array([-6, -1,   0, 1, -3,  2,  0, -1])
M = np.array([16, 10, -26, 8, 32, 30, 18, 28])
manwe = validation.PFV(data, K=K, M=M)
print(manwe.check_all())

misc other variables

In [ ]:
Qmax = data.h11+data.h21+4

# See if we can recreate the PFV

In [ ]:
Ks, Ms = Zp.coniZpM(
    data=data,
    ps=[manwe.pgrading[1:]],
    Qmin=Qmax,
    Qmax=Qmax,
    M0min=13,
    ellipsoid_dilation=20,
    max_N_pfvs=100_000_000,
    verbosity=0)

In [ ]:
print(Ks, Ms)

# Benchmark dilations

In [ ]:
benchmarking = True
dilations    = [1,2,3,4,5,7,10,13,16,20]#,25,30,35,40]
num_ps       = 100

Get some p-vectors

In [ ]:
if benchmarking:
    ps = Zp.pvecs(data, min_pts=num_ps, backend='gurobi')

Collect the dilation data

In [ ]:
if benchmarking:
    import time
    dilation = []
    num_pfvs = []
    times    = []
    
    for ellipsoid_dilation in dilations:
        print(ellipsoid_dilation,end='...\r')
        t0 = time.time()
        Ks, Ms = Zp.coniZpM(
            data=data,
            ps=ps,
            Qmin=Qmax,
            Qmax=Qmax,
            M0min=13,
            ellipsoid_dilation=ellipsoid_dilation,
            max_N_pfvs=100_000_000,
            verbosity=0)
        t1 = time.time()
    
        dilation.append(ellipsoid_dilation)
        num_pfvs.append(len(Ks))
        times.append(t1-t0)
    
    import pandas as pd
    df = pd.DataFrame({'dilation': dilation, 'num_pfvs':num_pfvs, 'time':times})

Plot the data

In [ ]:
if benchmarking:
    import matplotlib.pyplot as plt
    def round_sig(x, sig=2):
        return np.round(x, sig - np.floor(np.log10(np.abs(x))).astype(int) - 1)
    
    # make the figure
    fig,axs = plt.subplots(3,1, sharex=True)
    
    # raw data plots
    # --------------
    axs[0].scatter(df['dilation'], df['num_pfvs'])
    axs[1].scatter(df['dilation'], df['time'])
    axs[2].scatter(df['dilation'], df['num_pfvs']/df['time'])
    
    # fits
    # ----
    # #pfvs fit
    m,b = np.polyfit(
        x=df['dilation'][5:],
        y=df['num_pfvs'][5:],
        deg=1,
    )
    fit = m*df['dilation'] + b
    axs[0].plot(df['dilation'], fit,
                label=f'{round_sig(m)} * dilation + {round_sig(b)}')
    axs[0].legend()
    
    # time fit
    m,b = np.polyfit(
        x=np.log(df['dilation'][5:]),
        y=np.log(df['time'][5:]),
        deg=1,
    )
    fit = np.exp(b)*np.pow(df['dilation'],m)
    axs[1].plot(df['dilation'], fit,
                label=f'{round_sig(np.exp(b))} * dilation^{round_sig(m)}')
    axs[1].legend()
    
    #axs[0].set_yscale('log')
    #axs[1].set_yscale('log')
    #axs[2].set_yscale('log')
    
    axs[0].set_ylabel('# PFVs')
    axs[1].set_ylabel('time [s]')
    axs[2].set_ylabel('PFV rate [$s^{-1}$]')
    axs[2].set_xlabel('dilation')
    
    axs[0].set_title(f'Manwe, {len(ps)} p-vectors, dilation study')

# Scratch